<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Stacked Charts**


Estimated time needed: **45** minutes


In this lab, you will focus on visualizing data specifically using stacked charts. You will use SQL queries to extract the necessary data and apply stacked charts to analyze the composition and comparison within the data.


## Objectives


In this lab, you will perform the following:


- Visualize the composition of data using stacked charts.

- Compare multiple variables across different categories using stacked charts.

- Analyze trends within stacked chart visualizations.


## Setup: Downloading and Loading the Data
**Install the libraries**


In [ ]:
!pip install pandas

In [ ]:
!pip install matplotlib


**Download and Load the Data**


To start, download and load the dataset into a `pandas` DataFrame.



### Step 1: Download the dataset


In [ ]:
!wget -O survey-data.csv https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv

### Step 2: Import necessary libraries and load the dataset


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

### Load the data


In [ ]:
df = pd.read_csv("survey-data.csv")

### Display the first few rows of the data to understand its structure


In [ ]:
df.head()

### Task 1: Stacked Chart for Composition of Job Satisfaction Across Age Groups


##### 1. Stacked Chart of Median `JobSatPoints_6` and `JobSatPoints_7` for Different Age Groups


Visualize the composition of job satisfaction scores (`JobSatPoints_6` and `JobSatPoints_7`) across various age groups. This will help in understanding the breakdown of satisfaction levels across different demographics.



In [ ]:
# ---- Helpers used throughout this lab ----
import numpy as np

# Age is stored as text ranges, so map each range to an approximate number (its midpoint)
AGE_MAP = {
    'Under 18 years old': 17, '18-24 years old': 21, '25-34 years old': 29.5,
    '35-44 years old': 39.5, '45-54 years old': 49.5, '55-64 years old': 59.5,
    '65 years or older': 70,
}
AGE_ORDER = list(AGE_MAP.keys())

# TimeSearching / TimeAnswering are text ranges -> approximate minutes per day
TIME_MAP = {
    'Less than 15 minutes a day': 7.5, '15-30 minutes a day': 22.5,
    '30-60 minutes a day': 45, '60-120 minutes a day': 90, 'Over 120 minutes a day': 150,
}

def years_to_num(series):
    """YearsCodePro is text: turn 'Less than 1 year' / 'More than 50 years' into numbers."""
    return pd.to_numeric(series.replace({'Less than 1 year': '0', 'More than 50 years': '51'}),
                         errors='coerce')

def split_counts(series, top_n=None):
    """Count items in a ';'-separated multi-answer column."""
    counts = series.dropna().str.split(';').explode().str.strip().value_counts()
    return counts.head(top_n) if top_n else counts

for col in ['JobSatPoints_6', 'JobSatPoints_7', 'JobSat', 'ConvertedCompYearly']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

def stacked_share(multi_col, group_col, top_items=5, top_groups=None, title=''):
    """Stacked bar: share of each top item within each group (each bar adds to 100%)."""
    d = df[[multi_col, group_col]].dropna()
    if top_groups:
        d = d[d[group_col].isin(d[group_col].value_counts().head(top_groups).index)]
    d = d.assign(Item=d[multi_col].str.split(';')).explode('Item').reset_index(drop=True)
    top = d['Item'].value_counts().head(top_items).index
    d = d[d['Item'].isin(top)]
    ct = pd.crosstab(d[group_col], d['Item'], normalize='index') * 100
    if group_col == 'Age':
        ct = ct.reindex([a for a in AGE_ORDER if a in ct.index])
    ct.plot(kind='bar', stacked=True, figsize=(12, 6))
    plt.title(title)
    plt.ylabel('% within group')
    plt.xticks(rotation=30, ha='right')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# ---- 1.1 Median JobSatPoints_6 and JobSatPoints_7 by age group ----
by_age = df.groupby('Age')[['JobSatPoints_6', 'JobSatPoints_7']].median()
by_age = by_age.reindex([a for a in AGE_ORDER if a in by_age.index])
print(by_age)

by_age.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Median JobSatPoints_6 and JobSatPoints_7 by Age Group')
plt.xlabel('Age group')
plt.ylabel('Median points')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

##### Stacked Chart of `JobSatPoints_6` and `JobSatPoints_7` for Employment Status


Create a stacked chart to compare job satisfaction (`JobSatPoints_6` and `JobSatPoints_7`) across different employment statuses. This will show how satisfaction varies by employment type.


In [ ]:
# ---- 1.2 JobSatPoints_6 and JobSatPoints_7 by employment status (top 5) ----
top_emp = df['Employment'].value_counts().head(5).index
by_emp = df[df['Employment'].isin(top_emp)].groupby('Employment')[['JobSatPoints_6', 'JobSatPoints_7']].median()
print(by_emp)

by_emp.plot(kind='barh', stacked=True, figsize=(12, 6))
plt.title('Median JobSatPoints_6 and JobSatPoints_7 by Employment Status')
plt.xlabel('Median points')
plt.tight_layout()
plt.show()

### Task 2: Stacked Chart for Compensation and Job Satisfaction by Age Group


##### This stacked chart visualizes the composition of compensation (`ConvertedCompYearly`) and job satisfaction (`JobSatPoints_6`) specifically for respondents aged 30-35.


In [ ]:
# ---- 2.1 Compensation and job satisfaction for ages 30-35 ----
# Age is stored in ranges; 30-35 falls in '25-34 years old', the closest available group.
# Compensation is shown in $1,000s so it sits on a similar scale to the satisfaction points.
group = df[df['Age'] == '25-34 years old']
top_emp = group['Employment'].value_counts().head(5).index
summary = group[group['Employment'].isin(top_emp)].groupby('Employment').agg(
    Median_Comp_k=('ConvertedCompYearly', lambda s: s.median() / 1000),
    Median_JobSatPoints_6=('JobSatPoints_6', 'median'))
print(summary)

summary.plot(kind='barh', stacked=True, figsize=(12, 6))
plt.title('Ages 25-34: Median Compensation ($1,000s) and JobSatPoints_6 by Employment')
plt.xlabel('Value')
plt.tight_layout()
plt.show()

##### Stacked Chart of Median Compensation and Job Satisfaction Across Age Group


Compare the median compensation and job satisfaction metrics across different age groups. This helps visualize how compensation and satisfaction levels differ by age.


In [ ]:
# ---- 2.2 Median compensation and job satisfaction across age groups ----
by_age = df.groupby('Age').agg(
    Median_Comp_k=('ConvertedCompYearly', lambda s: s.median() / 1000),
    Median_JobSatPoints_6=('JobSatPoints_6', 'median'))
by_age = by_age.reindex([a for a in AGE_ORDER if a in by_age.index])
print(by_age)

by_age.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Median Compensation ($1,000s) and JobSatPoints_6 by Age Group')
plt.xlabel('Age group')
plt.ylabel('Value')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### Task 3: Comparing Data Using Stacked Charts


##### 1. Stacked Chart of Preferred Databases by Age Group




Visualize the top databases that respondents from different age groups wish to learn. Create a stacked chart to show the proportion of each database in each age group.


In [ ]:
# ---- 3.1 Preferred databases by age group ----
stacked_share('DatabaseWantToWorkWith', 'Age', top_items=5,
              title='Top 5 Databases Respondents Want to Learn, by Age Group')

##### 2. Stacked Chart of Employment Type by Job Satisfaction


Analyze the distribution of employment types within each job satisfaction level using a stacked chart. This will provide insights into how employment types are distributed across various satisfaction ratings.


In [ ]:
# ---- 3.2 Employment type within each job satisfaction level ----
d = df[['JobSat', 'Employment']].dropna()
top_emp = d['Employment'].value_counts().head(5).index
d = d[d['Employment'].isin(top_emp)]
ct = pd.crosstab(d['JobSat'], d['Employment'], normalize='index') * 100

ct.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Employment Type Distribution within Each Job Satisfaction Level')
plt.xlabel('JobSat (0-10)')
plt.ylabel('% of respondents at that level')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Task 4: Exploring Technology Preferences Using Stacked Charts


##### 1. Stacked Chart for Preferred Programming Languages by Age Group


Analyze how programming language preferences (`LanguageAdmired`) vary across age groups.


In [ ]:
# ---- 4.1 Admired languages by age group ----
stacked_share('LanguageAdmired', 'Age', top_items=5,
              title='Top 5 Admired Languages by Age Group')

##### 2. Stacked Chart for Technology Adoption by Employment Type


Explore how admired platforms (`PlatformAdmired`) differ across employment types (e.g., full-time, freelance)


In [ ]:
# ---- 4.2 Admired platforms by employment type ----
stacked_share('PlatformAdmired', 'Employment', top_items=5, top_groups=5,
              title='Top 5 Admired Platforms by Employment Type')

### Final Step: Review


In this lab, you focused on using stacked charts to understand the composition and comparison within the dataset. Stacked charts provided insights into job satisfaction, compensation, and preferred databases across age groups and employment types.


## Summary


After completing this lab, you will be able to:

- Use stacked charts to analyze the composition of data across categories, such as job satisfaction and compensation by age group.

- Compare data across different dimensions using stacked charts, enhancing your ability to communicate complex relationships in the data.

- Visualize distributions across multiple categories, such as employment type by satisfaction, to gain a deeper understanding of patterns within the dataset.


## Author:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


<!--
## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2024-10-28|1.2|Madhusudhan Moole|Updated lab|
|2024-10-16|1.1|Madhusudhan Moole|Updated lab|
|2024-10-15|1.0|Raghul Ramesh|Created lab|
--!>


Copyright © IBM Corporation. All rights reserved.
